In [ ]:
#pip install pywin32 openpyxl

In [ ]:
# ======================================================
# 0️⃣ IMPORT
# ======================================================

import os
import time
import shutil
from datetime import datetime
from openpyxl import load_workbook
import win32com.client

In [ ]:
# Configuration: set these environment variables before starting Jupyter.
SAP_PATH = os.environ.get("SAP_PATH", r"C:\Program Files (x86)\SAP\FrontEnd\SAPgui\saplogon.exe")
SAP_CONNECTION_NAME = os.environ["SAP_CONNECTION_NAME"]
USERNAME = os.environ["SAP_USERNAME"]
PASSWORD = os.environ["SAP_PASSWORD"]
TRANSACTION_CODE = os.environ["SAP_TRANSACTION_CODE"]

upload_path = os.environ["SAP_UPLOAD_PATH"]
log_archive_path = os.environ["SAP_LOG_ARCHIVE_PATH"]
non_fz_archive = os.environ["SAP_NON_FZ_ARCHIVE_PATH"]
fz_archive = os.environ["SAP_FZ_ARCHIVE_PATH"]

non_fz_success = os.path.join(non_fz_archive, "Completed")
non_fz_error = os.path.join(non_fz_archive, "Error")
fz_success = os.path.join(fz_archive, "Completed")
fz_error = os.path.join(fz_archive, "Error")
log_file = os.path.join(log_archive_path, "upload_log.txt")

for directory in (non_fz_success, non_fz_error, fz_success, fz_error, log_archive_path):
    os.makedirs(directory, exist_ok=True)


In [ ]:
# ======================================================
# 2️⃣ UTILITY FUNCTIONS
# ======================================================

def write_log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")
    print(message)


def wait_for_sap(session, timeout=30):
    start = time.time()
    while session.Busy:
        if time.time() - start > timeout:
            raise Exception("SAP Busy Timeout")
        time.sleep(0.5)


def check_cc2_column(file_path):
    try:
        wb = load_workbook(file_path, data_only=True)
        ws = wb.active
        value = ws["CC2"].value

        # ถ้าเป็น None หรือค่าว่าง ให้แสดง NONE
        if value is None or str(value).strip() == "":
            write_log("CC2 = NONE → NON-FREE ZONE")
            return False

        # ถ้ามีค่า
        if str(value).strip().upper() == "Y":
            write_log(f"CC2 = {value} → FREE ZONE")
            return True
        else:
            write_log(f"CC2 = {value} → NON-FREE ZONE")
            return False

    except Exception as e:
        write_log(f"ERROR reading CC2: {e}")
        return False

In [ ]:
# ======================================================
# 3️⃣ SAP CONNECTION
# ======================================================

def connect_sap():

    os.startfile(SAP_PATH)
    time.sleep(5)

    SapGuiAuto = win32com.client.GetObject("SAPGUI")
    application = SapGuiAuto.GetScriptingEngine

    connection = application.OpenConnection(SAP_CONNECTION_NAME, True)
    time.sleep(3)

    session = connection.Children(0)

    # Login
    session.findById("wnd[0]/usr/txtRSYST-BNAME").text = USERNAME
    session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = PASSWORD
    session.findById("wnd[0]").sendVKey(0)

    write_log("Logged into SAP successfully")

    # Enter Transaction
    session.findById("wnd[0]/tbar[0]/okcd").text = TRANSACTION_CODE
    session.findById("wnd[0]").sendVKey(0)

    write_log(f"Transaction {TRANSACTION_CODE} executed")

    return session


In [ ]:
# ======================================================
# 4️⃣ MODE SWITCH
# ======================================================

def switch_mode(session, new_mode, current_mode):

    if current_mode == new_mode:
        return current_mode

    if current_mode is not None:
        session.findById("wnd[0]/tbar[0]/btn[15]").press()
        wait_for_sap(session)

    if new_mode == "F":
        session.findById(
            "wnd[0]/usr/tabsTS_TAB/tabpTS_TAB1/"
            "ssubTS_TAB_SCA:ZFZ_EXIM10:0201/btnZFZ_004"
        ).press()
    else:
        session.findById(
            "wnd[0]/usr/tabsTS_TAB/tabpTS_TAB1/"
            "ssubTS_TAB_SCA:ZFZ_EXIM10:0201/btnZITOP081"
        ).press()

    wait_for_sap(session)
    return new_mode

In [ ]:
# ======================================================
# 5️⃣ UPLOAD
# ======================================================

def upload_file(session, file_path, mode):

    session.findById("wnd[0]/usr/ctxtPA_EXCEL").text = file_path
    wait_for_sap(session)

    session.findById("wnd[0]/tbar[1]/btn[8]").press()
    wait_for_sap(session)

    session.findById("wnd[0]/tbar[1]/btn[8]").press()
    wait_for_sap(session)

    # Handle popup only NON-FZ
    if mode == "NONF":
        try:
            session.findById("wnd[1]/usr/btnBUTTON_1").press()
            wait_for_sap(session)
        except:
            pass

In [ ]:
# ======================================================
# 6️⃣ MOVE FILES
# ======================================================

def move_files(excel_path, pdf_path, destination):

    if os.path.exists(excel_path):
        shutil.move(excel_path, os.path.join(destination, os.path.basename(excel_path)))

    if os.path.exists(pdf_path):
        shutil.move(pdf_path, os.path.join(destination, os.path.basename(pdf_path)))

In [ ]:
# ======================================================
# 7️⃣ MAIN PROCESS
# ======================================================

if __name__ == "__main__":

    session = connect_sap()

    start_time = time.time()
    write_log("========== START PROCESS ==========")

    excel_files = sorted(
        [f for f in os.listdir(upload_path) if f.lower().endswith(".xlsx")]
    )

    total_files = len(excel_files)
    success_count = 0
    error_count = 0
    missing_pdf_count = 0
    current_mode = None

    write_log(f"Total Excel files found: {total_files}")

    for file_name in excel_files:

        full_excel_path = os.path.join(upload_path, file_name)
        base_name = os.path.splitext(file_name)[0]
        pdf_name = base_name + ".pdf"
        full_pdf_path = os.path.join(upload_path, pdf_name)

        write_log(f"\nProcessing: {file_name}")

        try:

            cc2_result = check_cc2_column(full_excel_path)
            new_mode = "F" if cc2_result else "NONF"

            current_mode = switch_mode(session, new_mode, current_mode)

            upload_file(session, full_excel_path, new_mode)

            success_count += 1
            write_log("Upload SUCCESS")

            if not os.path.exists(full_pdf_path):
                missing_pdf_count += 1
                write_log("WARNING: PDF not found")

            destination = fz_success if cc2_result else non_fz_success
            move_files(full_excel_path, full_pdf_path, destination)

        except Exception as e:

            error_count += 1
            write_log(f"ERROR: {e}")

            destination = fz_error if cc2_result else non_fz_error
            move_files(full_excel_path, full_pdf_path, destination)

    duration = round(time.time() - start_time, 2)

    write_log("\n========== SUMMARY ==========")
    write_log(f"Total Files Found     : {total_files}")
    write_log(f"Upload Success        : {success_count}")
    write_log(f"Upload Failed         : {error_count}")
    write_log(f"Missing PDF Files     : {missing_pdf_count}")
    write_log(f"Total Processing Time : {duration} seconds")
    write_log("========== END PROCESS ==========")

    print("\n🎉 Process Completed.")